# Day 020：LoRA 注入、权重管理与训练更新

本 Notebook 对照 `model_lora.py` 和 `train_lora.py`，验证低秩增量、目标层注入、LoRA 权重保存/加载/合并，以及冻结原模型后的梯度更新。

In [ ]:
import sys
from pathlib import Path
candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(root for root in candidate_roots if (root / 'minimind' / 'model' / 'model_minimind.py').exists())
sys.path.insert(0, str(repo_root / 'minimind'))
import tempfile
import torch
from torch import nn, optim
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from model.model_lora import apply_lora, load_lora, save_lora, merge_lora
torch.manual_seed(0)
print('repo:', repo_root)

## 1. LoRA 的 A/B 低秩结构

`A: in_features -> rank`，`B: rank -> out_features`，LoRA 分支输出 `B(A(x))`。B 全零初始化时，应用 LoRA 不会立即改变原模型输出。

In [ ]:
config = MiniMindConfig(vocab_size=20, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=1, intermediate_size=16, max_position_embeddings=32, flash_attn=False)
model = MiniMindForCausalLM(config).eval()
input_ids = torch.tensor([[1, 4, 7]])
with torch.no_grad():
    before = model(input_ids).logits
apply_lora(model, rank=2)
targets = [name for name, module in model.named_modules() if hasattr(module, 'lora')]
with torch.no_grad():
    after = model(input_ids).logits
print('targets:', targets)
print('initial outputs equal:', torch.allclose(before, after))
print('one A/B shapes:', tuple(model.model.layers[0].self_attn.q_proj.lora.A.weight.shape), tuple(model.model.layers[0].self_attn.q_proj.lora.B.weight.shape))

## 2. save_lora / load_lora

LoRA 文件只保存带 `.lora.` 的 A/B 参数，不包含基础模型权重。加载前目标模型必须先执行 `apply_lora()`。

In [ ]:
with torch.no_grad():
    model.model.layers[0].self_attn.q_proj.lora.B.weight.fill_(0.25)
lora_path = Path(tempfile.gettempdir()) / 'day020-lora.pth'
save_lora(model, lora_path)
target = MiniMindForCausalLM(config)
apply_lora(target, rank=2)
before_load = target.model.layers[0].self_attn.q_proj.lora.B.weight.clone()
load_lora(target, lora_path)
after_load = target.model.layers[0].self_attn.q_proj.lora.B.weight
saved = torch.load(lora_path, map_location='cpu')
print('before load all zero:', torch.count_nonzero(before_load).item() == 0)
print('after load first value:', after_load.flatten()[0].item())
print('loaded equals source:', torch.equal(after_load, model.model.layers[0].self_attn.q_proj.lora.B.weight))
print('key count:', len(saved), 'all lora keys:', all('.lora.' in key for key in saved))
print('dtypes:', sorted({str(value.dtype) for value in saved.values()}))

## 3. merge_lora：`W + B @ A`

合并后保存的是普通模型权重；由于源码保存为 float16，运行时 LoRA 输出与合并输出应在容差内一致。

In [ ]:
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(4, 4, bias=False)
    @property
    def device(self):
        return next(self.parameters()).device

tiny = TinyModel()
apply_lora(tiny, rank=2)
with torch.no_grad():
    tiny.proj.lora.A.weight.copy_(torch.tensor([[1.,0.,0.,0.],[0.,1.,0.,0.]]))
    tiny.proj.lora.B.weight.copy_(torch.tensor([[1.,0.],[0.,2.],[3.,0.],[0.,4.]]))
x = torch.tensor([[2., 3., 5., 7.]])
with torch.no_grad():
    runtime = tiny.proj(x)
tiny_lora = Path(tempfile.gettempdir()) / 'day020-tiny-lora.pth'
tiny_merged = Path(tempfile.gettempdir()) / 'day020-tiny-merged.pth'
save_lora(tiny, tiny_lora)
merge_lora(tiny, tiny_lora, tiny_merged)
merged_state = torch.load(tiny_merged, map_location='cpu')
merged = nn.Linear(4, 4, bias=False)
merged.load_state_dict(merged_state)
with torch.no_grad():
    merged_output = merged(x)
print('outputs close:', torch.allclose(runtime, merged_output, atol=5e-3, rtol=1e-3))
print('max abs difference:', (runtime - merged_output).abs().max().item())
print('merged keys:', list(merged_state))
print('has lora keys:', any('.lora.' in key for key in merged_state))

## 4. 冻结原模型，只训练 LoRA

训练脚本把非 LoRA 参数设为 `requires_grad=False`，并只把 LoRA 参数传给 AdamW。

In [ ]:
train_model = MiniMindForCausalLM(config)
apply_lora(train_model, rank=2)
lora_params = []
for name, param in train_model.named_parameters():
    param.requires_grad = 'lora' in name
    if param.requires_grad:
        lora_params.append(param)
optimizer = optim.AdamW(lora_params, lr=1e-3)
print('total params:', sum(p.numel() for p in train_model.parameters()))
print('trainable params:', sum(p.numel() for p in train_model.parameters() if p.requires_grad))
print('optimizer params:', sum(p.numel() for group in optimizer.param_groups for p in group['params']))

## 5. backward 与梯度累积

第一次 backward 时 B 先得到梯度；设置 `accumulation_steps=2` 后第一步不更新，第二步才执行 optimizer.step()。

In [ ]:
b_param = train_model.model.layers[0].self_attn.q_proj.lora.B.weight
base_before = train_model.model.layers[0].self_attn.q_proj.weight.detach().clone()
for step in range(1, 3):
    outputs = train_model(torch.tensor([[1,4,7,2]]), labels=torch.tensor([[1,4,7,2]]))
    raw_loss = outputs.loss + outputs.aux_loss
    loss = raw_loss / 2
    loss.backward()
    print(f'step {step}: raw loss={raw_loss.item():.7f}, scaled loss={loss.item():.7f}, B grad norm={b_param.grad.norm().item():.7f}')
    if step % 2 == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        print('  optimizer step: yes')
    else:
        print('  optimizer step: no')
base_after = train_model.model.layers[0].self_attn.q_proj.weight.detach()
print('base q_proj unchanged:', torch.equal(base_before, base_after))
print('B nonzero:', torch.count_nonzero(b_param).item() > 0)

## 今日边界

已验证 LoRA 的低秩注入、适配器保存/加载/合并、冻结参数和梯度累积。Day 021 从基础模型 + LoRA 的推理一致性验收开始。